# Module 6.2: Object Detection & Defect Bounding Boxes
### 🇵🇱 Detekcja Obiektów i Ramki Ograniczające (Bounding Boxes)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/m06_l02_bounding_boxes.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Master bounding box parameterization [xmin, ymin, xmax, ymax] and understand multi-task classification + regression loss.  
> 🇵🇱 **Streszczenie:** Opanuj parametryzację ramek ograniczających i zrozum łączoną funkcję straty klasyfikacji i regresji pozycji.

- **Estimated Time:** 25 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **Spatial Localization**: In automated quality inspection, defect coordinates $[x_1, y_1, x_2, y_2]$ direct spot-repair robotics.
2. **Intersection over Union (IoU)**:
   $$\text{IoU} = \frac{\text{Area of Overlap}}{\text{Area of Union}}$$
3. **mAP@0.50 Threshold**: Standard industry metric requiring $\text{IoU} \ge 0.50$ for a detection to count as a True Positive.


In [ ]:
# 1. Bounding Box Math & Vectorized IoU Calculation
def compute_iou(box_a, box_b):
    x_left = max(box_a[0], box_b[0])
    y_top = max(box_a[1], box_b[1])
    x_right = min(box_a[2], box_b[2])
    y_bottom = min(box_a[3], box_b[3])
    
    inter_w = max(0.0, x_right - x_left)
    inter_h = max(0.0, y_bottom - y_top)
    inter_area = inter_w * inter_h
    
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union_area = area_a + area_b - inter_area
    return inter_area / union_area if union_area > 0 else 0.0

ground_truth = [100.0, 80.0, 260.0, 180.0]
predicted    = [115.0, 75.0, 275.0, 175.0]

iou = compute_iou(ground_truth, predicted)
print(f"Calculated IoU: {iou:.3f} ({(iou*100):.1f}%)")


In [ ]:
# 2. Visualizing Bounding Box Optical Inspection
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(8, 6))
metal = np.random.normal(0.4, 0.05, (300, 400))
ax.imshow(metal, cmap='gray', vmin=0, vmax=1)

gt_rect = patches.Rectangle((ground_truth[0], ground_truth[1]),
                            ground_truth[2] - ground_truth[0],
                            ground_truth[3] - ground_truth[1],
                            linewidth=2.5, edgecolor='#10b981', facecolor='none', label='Ground Truth')
ax.add_patch(gt_rect)

pred_rect = patches.Rectangle((predicted[0], predicted[1]),
                              predicted[2] - predicted[0],
                              predicted[3] - predicted[1],
                              linewidth=2.5, edgecolor='#f59e0b', linestyle='--', facecolor='none', 
                              label=f'CNN Prediction (IoU={iou:.2f})')
ax.add_patch(pred_rect)

plt.title("Industrial Optical Inspection - Bounding Box IoU Evaluation", fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.xlim(0, 400)
plt.ylim(300, 0)
plt.show()


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Parse Pascal VOC and YOLO format bounding box coordinate annotations *(🇵🇱 Wczytaj współrzędne ramek w formatach Pascal VOC oraz YOLO)*
- [ ] **Task 2:** Render bounding box overlays on defective sheet metal images *(🇵🇱 Wyrysuj ramki defektów na zdjęciach uszkodzonej blachy)*
- [ ] **Task 3:** Construct multi-loss target vectors (box coords + defect class) *(🇵🇱 Zbuduj wektory straty łączące błąd lokalizacji ramki z klasą wady)*


In [ ]:
# Exercise Challenge:
# Assert that the predicted box passes the quality control threshold (IoU >= 0.50)
passes_qc = compute_iou(ground_truth, predicted) >= 0.50
print(f"Quality Control Result: {'PASS (True Positive)' if passes_qc else 'REJECT'}")
assert passes_qc, "Predicted bounding box failed IoU threshold!"
print("✓ Computer vision evaluation exercise passed.")
